In [1]:
from messages_dataframe import load_dataframes
from pathlib import Path

frames = load_dataframes(Path("/home/henry/Documents/everything"))
qter_frames = load_dataframes()

In [2]:
import altair as alt
import polars as pl
import polars.selectors as cs

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
def messages_over_time(messages: pl.DataFrame) -> pl.DataFrame:
    """Count messages in calendar-month buckets for a temporal histogram."""
    return (
        messages.lazy()
        .filter(pl.col("timestamp").is_not_null())
        .filter(
            pl.col("channel").is_in(qter_frames.channels["channel_id"]).not_()
        )
        .with_columns(pl.col("timestamp").dt.truncate("1mo").alias("month"))
        .group_by("month", "author_id")
        .agg(pl.len().alias("message_count"))
        .join(frames.users.lazy(), left_on="author_id", right_on="user_id")
        .with_columns(
            pl.col("nicknames").list.first().alias("username"),
            pl.col("message_count")
            .sum()
            .over("author_id")
            .alias("total_message_count"),
            pl.col("message_count")
            .diff()
            .fill_null(pl.col("message_count"))
            .over("author_id", order_by="month")
            .alias("message_count_difference"),
        )
        .sort("month", "total_message_count", descending=[False, True])
        .collect()
    )

In [4]:
monthly_messages = messages_over_time(frames.messages)

(alt.Chart(monthly_messages)
.mark_area()
.encode(
    x=alt.X("month:T", title="Month"),
    y=alt.Y(
        "message_count:Q",
        title="Messages",
    ),
    color=alt.Color(
        "username:N",
        title="User",
        sort=alt.SortField(
            field="total_message_count", order="descending"
        ),
    ),
    order=alt.Order("total_message_count:Q", sort="descending"),
    tooltip=[
        alt.Tooltip("month:T", title="Month"),
        "username:N",
        "message_count:Q",
    ],
)
.properties(title="Discord messages over time (Without Qter)"))

/tmp/ipykernel_807701/4024806737.py:26: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


alt.Chart(...)

In [6]:
bottom = (pl.col("message_count_difference")
        .filter(pl.col("message_count_difference") < 0)
        .sum())

pos = bottom + pl.col("message_count_difference").abs().cum_sum()
neg = pos - pl.col("message_count_difference").abs()

monthly_messages = (
    messages_over_time(frames.messages)
    .sort("month", "message_count_difference")
    .with_columns(
        neg.over("month").alias("neg"),
        pos.over("month").alias("pos"),
    )
    .select("username", "month", "total_message_count", "message_count_difference", "pos", "neg")
)

monthly_messages

/tmp/ipykernel_807701/4024806737.py:26: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


username,month,total_message_count,message_count_difference,pos,neg
str,"datetime[μs, UTC]",u32,i64,i64,i64
"""o!""",2021-02-01 00:00:00 UTC,1591,5,5,0
"""o!""",2021-03-01 00:00:00 UTC,1591,-4,0,-4
"""bonrt""",2021-03-01 00:00:00 UTC,2,2,2,0
"""o!""",2021-04-01 00:00:00 UTC,1591,1,1,0
"""Bbokka""",2021-04-01 00:00:00 UTC,32,1,2,1
…,…,…,…,…,…
"""Damian""",2026-08-01 00:00:00 UTC,171,91,879,788
"""ericswpark""",2026-08-01 00:00:00 UTC,10826,102,981,879
"""Jack Hacker""",2026-08-01 00:00:00 UTC,26955,113,1094,981


In [8]:
(
    alt.Chart(monthly_messages)
    .mark_bar()
    .encode(
        x=alt.X("month:T", title="Month"),
        y=alt.Y(
            "neg:Q",
            title="Messages",
        ),
        y2="pos",
        color=alt.Color(
            "username:N",
            title="User",
            sort=alt.SortField(field="total_message_count", order="descending"),
        ),
        tooltip=[
            alt.Tooltip("month:T", title="Month"),
            "username:N",
            "message_count_difference:Q",
        ],
        #order=alt.Order("total_message_count:Q", sort="descending"),
    )
    .properties(
        title="Change in Discord messages over time (Without Qter)"
    )
)

alt.Chart(...)

In [ ]:
color_scale = alt.Scale(
    domain=[
        "Strongly disagree",
        "Disagree",
        "Neither agree nor disagree",
        "Agree",
        "Strongly agree",
    ],
    range=["#c30d24", "#f3a583", "#cccccc", "#94c6da", "#1770ab"],
)

y_axis = alt.Axis(title=None, offset=5, ticks=False, minExtent=60, domain=False)

alt.Chart(source).mark_bar().encode(
    x=alt.X("percentage_start:Q", title='Percentage'),
    x2="percentage_end:Q",
    y=alt.Y("question:N", axis=y_axis),
    color=alt.Color("type:N", title="Response", scale=color_scale),
)